# 07 — Atlas: which cell types age, across tissues?

Collects `05a` from every tissue that ran into one table, one row per tissue × cell type, and asks whether the Lung pattern (structural cells keep an age signal, immune cells are batch driven) holds across organs.

Every row uses `cross_age_all_splits`, the same batch-check rule in every tissue. The verdict uses the thresholds fixed in `06_survey` before any new tissue was run:

| Verdict | Rule |
| --- | --- |
| age signal | cross-age ≥ 0.7 and shuffled ≤ 0.6 |
| void | shuffled > 0.6: the pipeline reaches this from noise |
| partial | 0.5 ≤ cross-age < 0.7 |
| batch driven | cross-age < 0.5 |
| not testable | cross-age NaN (missing young or old age groups for this cell type) |

**Tissues share mice** (06: 23 mice in total, median 7 tissues each), so rows are not independent. Pass rates are reported per row, but every comparison is checked by leaving out one tissue at a time.

### 1. Setup

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact, mannwhitneyu
from src import aging as ag

A = ag.paths("atlas")

### 2. Collect every tissue's results

In [ ]:
rows = []
for d in sorted((ag.ROOT / "figure").iterdir()):
    f = d / "age_auroc.csv"
    if d.name == "atlas" or not f.exists():
        continue
    r = pd.read_csv(f, index_col=0)
    rep = pd.read_csv(d / "age_report.csv", index_col=0)
    r = r.join(rep[["auroc_mouse", "n_cells", "n_mice"]])
    r["tissue"] = d.name
    rows.append(r.rename_axis("cell_type").reset_index())
atlas = pd.concat(rows, ignore_index=True)
atlas["compartment"] = atlas["cell_type"].map(ag.compartment)
print(atlas["tissue"].value_counts())

### 3. Verdict per row

In [ ]:
def verdict(r):
    if pd.isna(r["cross_age_all_splits"]):
        return "not testable"
    if r["shuffled"] > 0.6:
        return "void"
    if r["cross_age_all_splits"] >= 0.7:
        return "age signal"
    return "partial" if r["cross_age_all_splits"] >= 0.5 else "batch driven"

atlas["verdict"] = atlas.apply(verdict, axis=1)
cols = ["tissue", "cell_type", "compartment", "logreg", "auroc_mouse", "shuffled",
        "cross_age_all_splits", "verdict", "n_mice", "n_cells"]
atlas = atlas[cols].sort_values(["compartment", "cross_age_all_splits"], ascending=[True, False])
atlas.round(3).to_csv(A.figure("atlas.csv"), index=False)
pd.crosstab(atlas["compartment"], atlas["verdict"], margins=True)

### 3b. Two checks on every row: technical batch, and dominant mice

**Technical batch.** The cross-age check only sees a batch that differs *between age groups*. If all young mice were processed one way and all old mice another, every split still passes (Bladder looked like this: all four cell types 0.89–0.96, leukocytes included). So each row is rerun with **only technical features** per cell (`ag.technical_adata`: log total counts, log genes per cell, ribosomal, dissociation and hemoglobin fractions). If those alone pass the cross-age check (`tech_cross_age` ≥ 0.7), the row is `tech_confounded`: its age signal can't be told apart from processing. Old cells can differ technically for real reasons, so this is a warning, not proof.

**Dominant mice.** A row whose cell type gets over half its cells from one mouse (`outlier_mice.csv` from 04) is rerun without that mouse. The main table keeps every mouse (the 300-cell cap already limits each); `flips` marks rows whose verdict changes without it.

In [ ]:
extra = []
for tissue in sorted(atlas["tissue"].unique()):
    a = sc.read_h5ad(ag.paths(tissue).processed)
    tech = ag.technical_adata(a)
    flags = pd.read_csv(ag.paths(tissue).figure("outlier_mice.csv"), index_col=0)
    for ct in atlas.loc[atlas["tissue"] == tissue, "cell_type"]:
        kw = ag.main_kw(ct, tissue)
        row = {"tissue": tissue, "cell_type": ct,
               "tech_auroc": ag.age_auc(tech, ct, **kw),
               "tech_cross_age": ag.cross_age_auc(tech, ct, **kw)}
        if ct in flags.index and flags.loc[ct, "top_mouse"] not in kw.get("drop_mice", []):
            m = flags.loc[ct, "top_mouse"]
            kw2 = {"drop_mice": kw.get("drop_mice", []) + [m]}
            row.update(dominant_mouse=m, dominant_share=flags.loc[ct, "share"],
                       logreg_without=ag.age_auc(a, ct, **kw2),
                       cross_age_without=ag.cross_age_auc(a, ct, **kw2))
        extra.append(row)
    del a, tech
atlas = atlas.merge(pd.DataFrame(extra), on=["tissue", "cell_type"], how="left")
atlas["tech_confounded"] = atlas["tech_cross_age"] >= 0.7
has_dom = atlas["dominant_mouse"].notna()
atlas["verdict_without"] = np.where(has_dom, atlas.assign(cross_age_all_splits=atlas["cross_age_without"])
                                    .apply(verdict, axis=1), None)
atlas["flips"] = has_dom & (atlas["verdict_without"] != atlas["verdict"])
atlas.round(3).to_csv(A.figure("atlas.csv"), index=False)


### 3c. Technical batch per tissue

Median of `tech_cross_age` over a tissue's rows, and how many rows are confounded.

In [ ]:
atlas.groupby("tissue").agg(rows=("cell_type", "size"),
                            median_cross_age=("cross_age_all_splits", "median"),
                            median_tech_cross_age=("tech_cross_age", "median"),
                            tech_confounded=("tech_confounded", "sum"),
                            age_signal=("verdict", lambda v: (v == "age signal").sum())).round(2)

### 3d. Rows dominated by one mouse, with and without it

In [ ]:
atlas.loc[has_dom, ["tissue", "cell_type", "dominant_mouse", "dominant_share", "logreg", "logreg_without",
                    "cross_age_all_splits", "cross_age_without", "verdict", "verdict_without", "flips"]].round(2)

### 4. Check the ambiguous labels against marker genes

Some TMS labels don't say what the cell is. Before they count toward a compartment, their mean expression of compartment markers is compared with clear cell types of the same tissue: stromal *Col1a1*, *Dcn*, *Pdgfra*; endothelial *Pecam1*, *Cdh5*; smooth muscle *Acta2*; immune *Ptprc*; epithelial *Epcam*, *Krt8*.

In [ ]:
markers = ["Col1a1", "Dcn", "Pdgfra", "Pecam1", "Cdh5", "Acta2", "Ptprc", "Epcam", "Krt8"]
check = {"Bladder": ["bladder cell", "bladder urothelial cell", "endothelial cell", "leukocyte"],
         "Kidney": ["kidney cortex artery cell", "brush cell", "fenestrated cell", "kidney mesangial cell", "macrophage"]}
tables = []
for tissue, cts in check.items():
    if not ag.paths(tissue).processed.exists():
        continue
    a = sc.read_h5ad(ag.paths(tissue).processed)
    a = a[a.obs["cell_ontology_class"].isin(cts), [g for g in markers if g in a.var_names]]
    m = sc.get.aggregate(a, by="cell_ontology_class", func="mean")
    tables.append(pd.DataFrame(m.layers["mean"], index=tissue + ": " + m.obs["cell_ontology_class"].astype(str),
                               columns=m.var_names))
marker_table = pd.concat(tables).round(2)
marker_table

### 5. Heatmap: cross-age AUROC, cell type × tissue

Rows grouped by compartment. A cell type name shared by several tissues (e.g. "endothelial cell", "macrophage") is one row. Grey = not run in that tissue. A dot marks a voided row (shuffled > 0.6).

In [ ]:
order = ["stromal", "endothelial", "epithelial", "other", "progenitor", "immune"]
atlas["comp_order"] = atlas["compartment"].map({c: i for i, c in enumerate(order)})
heat = atlas.pivot_table(index=["comp_order", "compartment", "cell_type"], columns="tissue",
                         values="cross_age_all_splits")
void = atlas.assign(v=atlas["verdict"].eq("void")).pivot_table(
    index=["comp_order", "compartment", "cell_type"], columns="tissue", values="v", aggfunc="max")
heat = heat.sort_index()
fig, ax = plt.subplots(figsize=(1 + 0.7 * heat.shape[1], 0.28 * heat.shape[0] + 1))
sns.heatmap(heat, cmap="RdBu_r", center=0.5, vmin=0, vmax=1, ax=ax, linewidths=0.3,
            cbar_kws={"label": "cross-age AUROC (all splits)"},
            yticklabels=[f"{c} [{comp}]" for _, comp, c in heat.index])
for (i, j) in zip(*np.where(void.reindex_like(heat).fillna(False).to_numpy().astype(bool))):
    ax.text(j + 0.5, i + 0.5, "•", ha="center", va="center", color="black")
ax.set_xlabel(""); ax.set_ylabel("")
ax.set_title("Age signal that survives the batch check (red = passes)")
fig.savefig(A.figure("atlas_cross_age.png"), dpi=200, bbox_inches="tight")

### 6. Pass rate per compartment

Voided and not-testable rows are left out. `rate` is the fraction of tissue × cell type rows with an age signal; `median_cross_age` uses every testable row.

In [ ]:
def rates(d):
    r = d.groupby("compartment").agg(rows=("verdict", "size"), tissues=("tissue", "nunique"),
                                     age_signal=("verdict", lambda v: (v == "age signal").sum()),
                                     median_cross_age=("cross_age_all_splits", "median"))
    r["rate"] = (r["age_signal"] / r["rows"]).round(2)
    return r.round(2).sort_values("rate", ascending=False)

use = atlas[~atlas["verdict"].isin(["void", "not testable"])]
clean = use[~use["tech_confounded"]]
print("all testable rows"); display(rates(use))
print("without technically confounded rows"); display(rates(clean))

### 7. Structural vs. immune: does the Lung pattern hold?

"Structural" = stromal + endothelial + epithelial. Two tests on the rows (Fisher on pass/fail, Mann-Whitney on cross-age AUROC), then the same comparison with each tissue left out in turn. Rows share mice, so the p-values are optimistic; the leave-one-tissue-out column says whether a single organ carries the result.

In [ ]:
def compare(d):
    s = d[d["compartment"].isin(["stromal", "endothelial", "epithelial"])]
    im = d[d["compartment"] == "immune"]
    table = [[(s["verdict"] == "age signal").sum(), (s["verdict"] != "age signal").sum()],
             [(im["verdict"] == "age signal").sum(), (im["verdict"] != "age signal").sum()]]
    return {"structural_rate": round(table[0][0] / max(len(s), 1), 2),
            "immune_rate": round(table[1][0] / max(len(im), 1), 2),
            "fisher_p": fisher_exact(table)[1] if len(s) and len(im) else np.nan,
            "mwu_p": mannwhitneyu(s["cross_age_all_splits"], im["cross_age_all_splits"])[1] if len(s) and len(im) else np.nan,
            "n_structural": len(s), "n_immune": len(im)}

loo = {"all tissues": compare(use)}
for t in sorted(use["tissue"].unique()):
    loo[f"without {t}"] = compare(use[use["tissue"] != t])
loo["all, no tech-confounded rows"] = compare(clean)
for t in sorted(clean["tissue"].unique()):
    loo[f"no tech-confounded, without {t}"] = compare(clean[clean["tissue"] != t])
loo = pd.DataFrame(loo).T
loo.to_csv(A.figure("atlas_structural_vs_immune.csv"))
loo

### 8. The same cell type in several tissues

A cell type name present in 3+ tissues: does it pass everywhere, nowhere, or depend on the organ? These rows share mice across tissues, so agreement can come from a mouse-level batch effect as well as from shared biology.

In [ ]:
multi = atlas.groupby("cell_type")["tissue"].nunique()
shared = atlas[atlas["cell_type"].isin(multi[multi >= 3].index)]
shared.pivot_table(index="cell_type", columns="tissue", values="cross_age_all_splits").round(2)

### 9. Rows with an age signal

The rows to carry into gene analysis (08–09).

In [ ]:
passing = atlas[atlas["verdict"] == "age signal"].sort_values("cross_age_all_splits", ascending=False)
passing.round(2)

### Findings

70 tissue × cell type rows from 9 tissues (23 mice in total; tissues share mice). Thresholds as fixed in 06.

- **The structural-over-immune pattern holds across organs.** Rows with an age signal: structural (stromal, endothelial, epithelial) 0.69, immune 0.32 (Fisher p = 0.013, Mann–Whitney on cross-age p = 0.001). Dropping the 8 technically confounded rows sharpens it: 0.75 vs 0.33 (p = 0.008 and 0.0009).
- **It is not carried by one tissue, but Lung, Kidney and Marrow matter most.** Without the technically confounded rows, leaving out any one tissue keeps the Mann–Whitney p below 0.05 (largest 0.045, without Lung); with those rows in, it is 0.087 without Lung. The pass/fail Fisher p rises to 0.06–0.09 without Lung, Kidney or Marrow. Read as: cross-age AUROC is reliably higher in structural cells once technical batch is removed; the binary pass-rate difference is suggestive with this many rows.
- **Immune cells are not batch driven everywhere.** In Limb_Muscle, B cells (0.83), T cells (0.86) and macrophages (0.76) pass, as do Spleen T cells. The Lung result (immune at ~0.5) is partly a Lung and Marrow property.
- **Technical batch is real in Bladder.** Technical features alone (depth, genes per cell, ribosomal, dissociation, blood fractions) pass the cross-age check in 2 of 4 Bladder rows (median 0.75), so Bladder's uniform 0.89–0.96 is partly processing. Heart fibroblasts (0.74) are the only other age-signal row flagged. 26 of 29 age-signal rows survive this check.
- **Dominant mice change two rows, neither of them a main-table pass.** Without 1-M-63, Liver sinusoidal endothelium goes from partial (0.57) to an age signal (0.82); without 30-M-2, Heart coronary-artery endothelium goes from partial (0.53) to batch driven (0.49). 30-M-2, excluded from Lung myeloid cells, also dominates cell types in Heart and Marrow, which fits a body-wide condition in that mouse.
- **Labels check out against markers:** "bladder cell" is mesenchymal (*Col1a1*, *Dcn*), "kidney cortex artery cell" endothelial (*Pecam1*, *Cdh5*), "brush cell" epithelial (*Krt8*, *Epcam*), mesangial cells smooth-muscle-like (*Acta2*).
- **Not testable:** Fat, Pancreas, Skin, Large_Intestine (no young mice), Thymus (one young mouse), Trachea (no old mice), Mammary_Gland (young are all 3m).